In [0]:
-- Dimensão Clientes
CREATE OR REPLACE TABLE gold.dim_clientes AS
SELECT DISTINCT
  customer_id AS id_cliente,
  customer_unique_id AS id_cliente_unico,
  cidade,
  estado
FROM silver.customers;

SELECT * FROM gold.dim_clientes;

In [0]:
-- Dimensão Produtos
CREATE OR REPLACE TABLE gold.dim_produtos AS
SELECT DISTINCT
  product_id AS id_produto,
  categoria_produto
FROM silver.products;

SELECT * FROM gold.dim_produtos;

In [0]:
-- Dimensão Avaliações Enriquecidas por IA (1a avaliação mais recente por pedido para manter relação 1:N)
CREATE OR REPLACE TABLE gold.dim_avaliacoes_ia AS
WITH avaliacoes_deduplicadas AS (
  SELECT
    order_id AS id_pedido,
    review_id AS id_avaliacao,
    nota_avaliacao,
    sentimento,
    categoria_problema,
    resumo_curto,
    comentario,
    ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY data_avaliacao DESC) AS rn  -- OVER anuncia Window Function
  FROM silver.reviews_enriquecidas_ia
)
SELECT
  id_pedido,
  id_avaliacao,
  nota_avaliacao,
  sentimento,
  categoria_problema,
  resumo_curto,
  comentario
FROM avaliacoes_deduplicadas
WHERE rn = 1;
/* Leitura:
CREATE TABLE ... AS: WITH ... AS [CTE], SELECT ... FROM [CTE] WHERE ...
Crie uma tabela A contendo, a partir do conjunto de dados temporário B, as colunas tal a partir do conjunto B onde C.
*/

SELECT * FROM gold.dim_avaliacoes_ia;

In [0]:
-- Dimensão Calendário (gerada dinamicamente a partir do intervalo de compras)
CREATE OR REPLACE TABLE gold.dim_calendario AS
WITH limites AS (
  SELECT
    MIN(CAST(data_compra AS DATE)) AS data_min,
    MAX(CAST(data_compra AS DATE)) AS data_max
  FROM silver.orders
),
datas AS (
  SELECT EXPLODE(SEQUENCE(data_min, data_max, INTERVAL 1 DAY)) AS data
  FROM limites
)
SELECT
  data,
  YEAR(data) AS ano,
  MONTH(data) AS mes_num,
  DATE_FORMAT(data, 'MMM/yy') AS mes_ano,
  QUARTER(data) AS trimestre,
  DAYOFWEEK(data) AS dia_semana_num
FROM datas;

SELECT * FROM gold.dim_calendario;

In [0]:
-- Tabela fato única que junta orders, order_items e dim_avaliacoes_ia
CREATE OR REPLACE TABLE gold.fact_vendas AS
SELECT
  i.order_id AS id_pedido,
  i.order_item_id AS id_item_pedido,
  o.customer_id AS id_cliente,
  i.product_id AS id_produto,
  CAST(o.data_compra AS DATE) AS data_compra,
  o.order_status AS status_pedido,
  o.status_sla,
  o.dias_diferenca_entrega,
  i.valor_produto,
  i.valor_frete,
  (i.valor_produto + i.valor_frete) AS valor_total_item,
  CASE WHEN ia.id_pedido IS NOT NULL THEN 1 ELSE 0 END AS flag_avaliado_ia
FROM silver.order_items i
INNER JOIN silver.orders o
  ON i.order_id = o.order_id
LEFT JOIN gold.dim_avaliacoes_ia ia
  ON i.order_id = ia.id_pedido;

SELECT * FROM gold.fact_vendas LIMIT 500;